# Metadata

Domain: Material Science

Subdomain: Crystallography

References: https://doi.org/10.1093/biomet/45.1-2.229 (Mackenzie, Biometrika 45, 229-240, 1958); https://doi.org/10.1016/0001-6160(66)90168-4 (Brandon, Acta Metallurgica 14, 1479-1484, 1966); https://doi.org/10.1107/S056773947400043X (Grimmer, Bollmann and Warrington, Acta Cryst. A30, 197-207, 1974)


# Title

Random-Texture Statistics of Cubic Grain Boundaries: Exact Misorientation-Ball Overlaps, the Mackenzie Distribution and Brandon CSL Fractions


# Subproblem 1

## Prompt

In texture analysis and grain-boundary statistics, crystal orientations and misorientations are elements of the rotation group \\(SO(3)\\), and randomly oriented grains are described by the uniform (Haar) probability measure \\(\\mu\\) on \\(SO(3)\\), normalized such that \\(\\mu(SO(3))=1\\). A rotation is represented by a unit quaternion \\(q=(w,x,y,z)=(\\cos\\tfrac{\\omega}{2},\\,\\mathbf{n}\\sin\\tfrac{\\omega}{2})\\), where \\(\\omega\\in[0^\\circ,180^\\circ]\\) is the rotation angle about the unit axis \\(\\mathbf{n}\\); \\(q\\) and \\(-q\\) describe the same rotation. The distance \\(d(R_1,R_2)\\in[0^\\circ,180^\\circ]\\) between two rotations is the rotation angle of \\(R_1R_2^{-1}\\). For a rotation \\(R\\) and a radius \\(r\\in[0^\\circ,180^\\circ]\\), the ball of rotations within \\(r\\) of \\(R\\) is

\\[
B(R,r)=\\{Q\\in SO(3):\\ d(Q,R)\\le r\\}.
\\]

Such balls are used as tolerance regions around special misorientations, and their overlaps determine how often a random grain boundary satisfies two tolerance criteria at the same time.

Implement a function that returns \\(\\mu\\big(B(R_1,r_1)\\cap B(R_2,r_2)\\big)\\). The rotations are given by unit quaternions with arbitrary signs and the radii in degrees. The result must have an absolute error below \\(10^{-12}\\) for all radii in \\([0^\\circ,180^\\circ]\\). Use only NumPy and the Python standard library.

```python
import numpy as np

def ball_overlap_measure(q1, q2, r1, r2):
    """
    Haar measure of the intersection of two balls of rotations.

    Inputs:
        q1, q2 : array_like of 4 floats
            Unit quaternions (w, x, y, z) of the rotations R1 and R2. q and -q denote the same rotation.
        r1, r2 : float
            Radii of the balls B(R1, r1) and B(R2, r2) in degrees, 0 <= r <= 180.

    Output:
        measure : float
            mu(B(R1, r1) ∩ B(R2, r2)), with mu the Haar probability measure on SO(3) (mu(SO(3)) = 1).
            Dimensionless; absolute error below 1e-12.
    """
    ...
```


## Background

**Haar measure.** The Haar measure \\(\\mu\\) is invariant under multiplication of all rotations from the left or from the right by a fixed rotation. The rotation angle \\(\\omega\\) of a random rotation has the probability density \\((1-\\cos\\omega)/\\pi\\) on \\([0,\\pi]\\), so a single ball has the measure \\(\\mu(B(R,r))=(r-\\sin r)/\\pi\\) (with \\(r\\) in radians). Because of the invariance, \\(\\mu\\big(B(R_1,r_1)\\cap B(R_2,r_2)\\big)\\) depends only on \\(d(R_1,R_2)\\), \\(r_1\\) and \\(r_2\\).

**Quaternions.** Unit quaternions form the unit sphere \\(S^3\\subset\\mathbb{R}^4\\), and the map \\(q\\mapsto R\\) is two-to-one onto \\(SO(3)\\). The Haar measure on \\(SO(3)\\) corresponds to the uniform measure on \\(S^3\\). The distance between two rotations satisfies \\(\\cos\\big(d(R_1,R_2)/2\\big)=|q_1\\cdot q_2|\\), where the absolute value accounts for the two quaternions \\(\\pm q\\) of each rotation.

**Special cases.** Balls whose centres are more than \\(r_1+r_2\\) apart do not intersect, a ball of radius \\(180^\\circ\\) is the whole group, and \\(B(R_2,r_2)\\subseteq B(R_1,r_1)\\) whenever \\(d(R_1,R_2)+r_2\\le r_1\\).

References: J. K. Mackenzie, Biometrika 45, 229-240 (1958), https://doi.org/10.1093/biomet/45.1-2.229


## Testing Template

In [ ]:
import math
import numpy as np


def _quat(axis, angle_deg):
    n = np.asarray(axis, dtype=float)
    n = n / np.linalg.norm(n)
    t = math.radians(angle_deg) / 2.0
    return np.array([math.cos(t), *(math.sin(t) * n)])


def _qmul(a, b):
    w1, x1, y1, z1 = a
    w2, x2, y2, z2 = b
    return np.array([w1 * w2 - x1 * x2 - y1 * y2 - z1 * z2, w1 * x2 + x1 * w2 + y1 * z2 - z1 * y2,
                     w1 * y2 - x1 * z2 + y1 * w2 + z1 * x2, w1 * z2 + x1 * y2 - y1 * x2 + z1 * w2])


def _ball(r_deg):
    r = math.radians(r_deg)
    return (r - math.sin(r)) / math.pi


def test_case_1():
    q = _quat([1, 2, 3], 37.0)
    # identical centres
    for r in [0.0, 1.0, 15.0, 60.0, 120.0, 180.0]:
        assert abs(ball_overlap_measure(q, q, r, r) - _ball(r)) < 1e-12
        assert abs(ball_overlap_measure(q, -q, r, r) - _ball(r)) < 1e-12
    assert abs(ball_overlap_measure(q, q, 20.0, 70.0) - _ball(20.0)) < 1e-12
    # nested balls
    q2 = _qmul(q, _quat([1, -1, 2], 20.0))
    assert abs(ball_overlap_measure(q, q2, 30.0, 5.0) - _ball(5.0)) < 1e-12
    assert abs(ball_overlap_measure(-q2, q, 5.0, 30.0) - _ball(5.0)) < 1e-12
    assert abs(ball_overlap_measure(q, q2, 25.0, 5.0) - _ball(5.0)) < 1e-12
    # disjoint and touching balls
    q3 = _qmul(q, _quat([0, 1, 1], 60.0))
    assert abs(ball_overlap_measure(q, q3, 20.0, 30.0)) < 1e-12
    assert abs(ball_overlap_measure(q, -q3, 25.0, 35.0)) < 1e-12
    # a ball of radius 180 degrees is the whole group
    for r in [0.0, 35.0, 170.0]:
        assert abs(ball_overlap_measure(q, q3, 180.0, r) - _ball(r)) < 1e-12
        assert abs(ball_overlap_measure(q3, q, r, 180.0) - _ball(r)) < 1e-12


def test_case_2():
    # partially overlapping balls; R2 = R1 * (rotation by omega about [1, 2, 3])
    base = _quat([2, -1, 1], 71.0)
    cases = [
        (40.0, 30.0, 25.0, 0.00059148218403882026),
        (90.0, 60.0, 50.0, 0.0022586712657167275),
        (60.0, 45.0, 45.0, 0.0038548887015610936),
        (100.0, 70.0, 40.0, 0.0005461875762406918),
        (150.0, 90.0, 80.0, 0.0039732452898518054),
    ]
    for omega, r1, r2, expected in cases:
        q2 = _qmul(base, _quat([1, 2, 3], omega))
        assert abs(ball_overlap_measure(base, q2, r1, r2) - expected) < 1e-12
        assert abs(ball_overlap_measure(q2, -base, r2, r1) - expected) < 1e-12
    # invariance under a common rotation from the left
    g = _quat([3, 0, -1], 123.0)
    q2 = _qmul(base, _quat([1, 2, 3], 90.0))
    assert abs(ball_overlap_measure(_qmul(g, base), _qmul(g, q2), 60.0, 50.0) - 0.0022586712657167275) < 1e-12


def test_case_3():
    # large balls: parts of the intersection lie on both sides of the double cover
    base = _quat([2, -1, 1], 71.0)
    cases = [
        (170.0, 100.0, 100.0, 0.012244441820474994),
        (120.0, 170.0, 150.0, 0.59651327103954438),
        (179.0, 179.0, 179.0, 0.97787530584247838),
        (180.0, 90.0, 95.0, 0.00061415757985357302),
        (180.0, 120.0, 120.0, 0.10031432409533196),
        (135.0, 160.0, 140.0, 0.43504224664275474),
        (100.0, 180.0, 20.0, 0.002242718216553724),
    ]
    for omega, r1, r2, expected in cases:
        q2 = _qmul(base, _quat([1, 2, 3], omega))
        assert abs(ball_overlap_measure(base, q2, r1, r2) - expected) < 1e-12
        assert abs(ball_overlap_measure(-q2, base, r2, r1) - expected) < 1e-12


## Solution

In [ ]:
import math

import numpy as np


def ball_overlap_measure(q1, q2, r1, r2):
    q1 = np.asarray(q1, dtype=float)
    q2 = np.asarray(q2, dtype=float)
    q1 = q1 / np.linalg.norm(q1)
    q2 = q2 / np.linalg.norm(q2)
    dot = float(q1 @ q2)
    if dot < 0.0:
        q2, dot = -q2, -dot
    # angle between the two quaternions on S^3 (half the rotation angle between the rotations), in [0, pi/2]
    delta = math.atan2(float(np.linalg.norm(q1 - dot * q2)), dot)
    alpha = 0.5 * math.radians(r1)
    beta = 0.5 * math.radians(r2)

    # The uniform measure on S^3 projects onto the uniform measure on the unit disk of any 2-plane. In the plane spanned
    # by q1 and q2, the cap {x.q1 >= cos(alpha)} becomes the disk segment {u >= cos(alpha)} and the cap around +-q2 the
    # segment {u cos(t) + v sin(t) >= cos(beta)} with t = delta (for +q2) or t = pi - delta (for -q2).
    def half_plane_clip(poly, nx, ny, c):  # keep nx*u + ny*v >= c
        out = []
        for k in range(len(poly)):
            p, q = poly[k], poly[(k + 1) % len(poly)]
            fp = nx * p[0] + ny * p[1] - c
            fq = nx * q[0] + ny * q[1] - c
            if fp >= 0.0:
                out.append(p)
            if (fp >= 0.0) != (fq >= 0.0):
                s = fp / (fp - fq)
                out.append((p[0] + s * (q[0] - p[0]), p[1] + s * (q[1] - p[1])))
        return out

    def disk_triangle_area(a, b):  # signed area of (unit disk) ∩ triangle(origin, a, b)
        dx, dy = b[0] - a[0], b[1] - a[1]
        qa = dx * dx + dy * dy
        if qa == 0.0:
            return 0.0
        qb = a[0] * dx + a[1] * dy
        qc = a[0] * a[0] + a[1] * a[1] - 1.0
        disc = qb * qb - qa * qc
        if disc <= 0.0:  # the line does not enter the open disk: the whole piece lies outside
            return 0.5 * math.atan2(a[0] * b[1] - a[1] * b[0], a[0] * b[0] + a[1] * b[1])
        cuts = [0.0]
        if disc > 0.0:
            root = math.sqrt(disc)
            for t in ((-qb - root) / qa, (-qb + root) / qa):
                if 0.0 < t < 1.0:
                    cuts.append(t)
        cuts.append(1.0)
        area = 0.0
        for t0, t1 in zip(cuts[:-1], cuts[1:]):
            p = (a[0] + t0 * dx, a[1] + t0 * dy)
            q = (a[0] + t1 * dx, a[1] + t1 * dy)
            tm = 0.5 * (t0 + t1)
            mx, my = a[0] + tm * dx, a[1] + tm * dy
            cross = p[0] * q[1] - p[1] * q[0]
            if mx * mx + my * my <= 1.0:
                area += 0.5 * cross
            else:
                area += 0.5 * math.atan2(cross, p[0] * q[0] + p[1] * q[1])
        return area

    def region_area(t):
        poly = [(-2.0, -2.0), (2.0, -2.0), (2.0, 2.0), (-2.0, 2.0)]
        poly = half_plane_clip(poly, 1.0, 0.0, math.cos(alpha))
        if poly:
            poly = half_plane_clip(poly, math.cos(t), math.sin(t), math.cos(beta))
        if len(poly) < 3:
            return 0.0
        return abs(sum(disk_triangle_area(poly[k], poly[(k + 1) % len(poly)]) for k in range(len(poly))))

    # both caps around +q1 and -q1 are counted: factor 2; S^3 measure = disk area / pi
    return 2.0 * (region_area(delta) + region_area(math.pi - delta)) / math.pi


# Subproblem 2

## Prompt

For a crystal with cubic symmetry, the 24 proper rotations that map a cube onto itself form the group \\(G\\): the identity, the rotations by \\(\\pm90^\\circ\\) and \\(180^\\circ\\) about the three cube axes \\(\\langle100\\rangle\\), the rotations by \\(\\pm120^\\circ\\) about the four body diagonals \\(\\langle111\\rangle\\), and the rotations by \\(180^\\circ\\) about the six face diagonals \\(\\langle110\\rangle\\). The misorientation \\(M\\) between two cubic grains has the crystallographically equivalent descriptions \\(O_1MO_2\\) with \\(O_1,O_2\\in G\\), and its disorientation angle is the smallest rotation angle among them,

\\[
\\theta_{\\mathrm{dis}}(M)=\\min_{O_1,O_2\\in G}\\ \\omega(O_1MO_2),
\\]

where \\(\\omega(\\cdot)\\in[0^\\circ,180^\\circ]\\) denotes the rotation angle.

For grains with uncorrelated orientations (random texture), \\(M\\) is distributed according to the uniform (Haar) probability measure \\(\\mu\\) on \\(SO(3)\\), normalized such that \\(\\mu(SO(3))=1\\). Implement a function that returns the cumulative distribution function of the disorientation angle,

\\[
F(\\theta)=\\mu\\big(\\{M\\in SO(3):\\ \\theta_{\\mathrm{dis}}(M)\\le\\theta\\}\\big),
\\]

for a given \\(\\theta\\) in degrees, \\(0\\le\\theta\\le180\\). The result must have an absolute error below \\(10^{-10}\\) for every \\(\\theta\\) in this range. Use only NumPy and the Python standard library.

```python
import numpy as np

def cubic_disorientation_cdf(theta):
    """
    Cumulative distribution function of the disorientation angle of randomly misoriented cubic grains.

    Input:
        theta : float
            Disorientation angle in degrees, 0 <= theta <= 180.

    Output:
        F : float
            Probability that a Haar-random misorientation M has theta_dis(M) <= theta, where theta_dis is the
            smallest rotation angle among the equivalent descriptions O1 M O2 (O1, O2 in the 24-element proper
            rotation group of the cube). Dimensionless; absolute error below 1e-10.
    """
    ...
```


## Background

**The Mackenzie distribution.** The distribution of the disorientation angle for randomly oriented cubic grains was derived by Mackenzie (1958). It is the reference against which measured misorientation distributions (for example from EBSD maps) are compared to detect texture or special grain-boundary populations, and it differs strongly from the distribution of the rotation angle itself because the 24 symmetry-equivalent descriptions allow the smallest angle to be chosen.

**Haar measure and rotation angles.** The Haar measure is invariant under multiplication from the left and from the right by fixed rotations, and the rotation angle is invariant under conjugation, \\(\\omega(QMQ^{-1})=\\omega(M)\\). The rotation angle of a Haar-random rotation has the probability density \\((1-\\cos\\omega)/\\pi\\) on \\([0,\\pi]\\), so the set of rotations with rotation angle at most \\(r\\) has the measure \\((r-\\sin r)/\\pi\\) (with \\(r\\) in radians). A rotation by \\(\\omega\\) about the unit axis \\(\\mathbf{n}\\) corresponds to the unit quaternions \\(\\pm(\\cos\\tfrac{\\omega}{2},\\mathbf{n}\\sin\\tfrac{\\omega}{2})\\), and the Haar measure corresponds to the uniform measure on the unit sphere \\(S^3\\) of quaternions. Equivalently, a rotation can be described by its Rodrigues vector \\(\\mathbf{n}\\tan\\tfrac{\\omega}{2}\\).

**Limits.** \\(F(0)=0\\), \\(F\\) is non-decreasing, and \\(F(\\theta)=1\\) for all angles at or above the largest possible disorientation angle of cubic grains.

References: J. K. Mackenzie, Biometrika 45, 229-240 (1958), https://doi.org/10.1093/biomet/45.1-2.229


## Testing Template

In [ ]:
import math


def test_case_1():
    # small angles: the rotations within theta of different symmetry elements do not overlap
    for theta in [0.0, 0.5, 5.0, 15.0, 30.0, 40.0, 45.0]:
        t = math.radians(theta)
        assert abs(cubic_disorientation_cdf(theta) - 24.0 * (t - math.sin(t)) / math.pi) < 1e-10
    # monotonic and bounded
    values = [cubic_disorientation_cdf(0.5 * k) for k in range(0, 361, 7)]
    assert all(b >= a - 1e-12 for a, b in zip(values, values[1:]))
    assert all(-1e-12 <= v <= 1.0 + 1e-12 for v in values)


def test_case_2():
    # intermediate angles
    cases = [
        (46.0, 0.6365159050652123),
        (48.5, 0.7264893856490133),
        (50.0, 0.7758566679457417),
        (53.0, 0.8627673510980809),
        (56.5, 0.9415150953189559),
        (59.0, 0.9808473648811813),
        (60.0, 0.992280125488668),
        (60.5, 0.9963573413402229),
        (60.72, 0.9975130325490773),
    ]
    for theta, expected in cases:
        assert abs(cubic_disorientation_cdf(theta) - expected) < 1e-10


def test_case_3():
    # angles close to the largest disorientation angle
    cases = [
        (60.8, 0.9978393994226368),
        (61.2, 0.9989880423818288),
        (61.7, 0.9996972916277741),
        (62.2, 0.9999540616336734),
        (62.6, 0.9999983827022721),
        (62.78, 0.9999999985319115),
        (62.795, 0.9999999999826239),
    ]
    for theta, expected in cases:
        assert abs(cubic_disorientation_cdf(theta) - expected) < 1e-10
    for theta in [62.8, 63.0, 75.0, 90.0, 135.0, 180.0]:
        assert abs(cubic_disorientation_cdf(theta) - 1.0) < 1e-10


## Solution

In [ ]:
import math

import numpy as np


def cubic_disorientation_cdf(theta):
    theta = float(theta)
    if theta <= 0.0:
        return 0.0
    if theta >= 180.0:
        return 1.0

    # Fundamental zone of the 24 proper cubic rotations in Rodrigues space r = n tan(omega/2): a truncated cube bounded by
    # |r_i| <= tan(pi/8) (from the 90 degree rotations) and |r_1| + |r_2| + |r_3| <= 1 (from the 120 degree rotations).
    t8 = math.tan(math.pi / 8.0)
    normals, dists = [], []
    for k in range(3):
        for sgn in (1.0, -1.0):
            n = np.zeros(3)
            n[k] = sgn
            normals.append(n)
            dists.append(t8)
    for sx in (1.0, -1.0):
        for sy in (1.0, -1.0):
            for sz in (1.0, -1.0):
                normals.append(np.array([sx, sy, sz]) / math.sqrt(3.0))
                dists.append(1.0 / math.sqrt(3.0))
    normals = np.array(normals)
    dists = np.array(dists)
    cos_pairs = np.clip(normals @ normals.T, -1.0, 1.0)
    rho_edge = math.sqrt(2.0) * t8                              # nearest points of the edges
    rho_vertex = math.sqrt(2.0 * t8 * t8 + (1.0 - 2.0 * t8) ** 2)  # vertices (disorientation 62.80 degrees)

    def inside_fraction(rho):
        # fraction of the sphere |r| = rho inside the zone: caps beyond each face, corrected by the pairwise overlaps of
        # caps (edges); for rho below the vertex distance no point of the sphere lies beyond three faces
        if rho >= rho_vertex:
            return 0.0
        cap = np.arccos(np.clip(dists / rho, -1.0, 1.0))        # angular radii of the caps (0 if the face is not reached)
        f = 1.0 - np.sum(0.5 * (1.0 - np.cos(cap)))
        m = len(dists)
        for i in range(m):
            for j in range(i + 1, m):
                a, b = cap[i], cap[j]
                if a <= 0.0 or b <= 0.0:
                    continue
                c = math.acos(cos_pairs[i, j])
                if c >= a + b:
                    continue
                if c <= abs(a - b):
                    lens = 2.0 * math.pi * (1.0 - math.cos(min(a, b)))
                else:
                    x = (math.cos(c) - math.cos(a) * math.cos(b)) / (math.sin(a) * math.sin(b))
                    y = (math.cos(b) - math.cos(c) * math.cos(a)) / (math.sin(c) * math.sin(a))
                    z = (math.cos(a) - math.cos(c) * math.cos(b)) / (math.sin(c) * math.sin(b))
                    lens = 2.0 * (math.pi - math.acos(max(-1.0, min(1.0, x)))
                                  - math.cos(a) * math.acos(max(-1.0, min(1.0, y)))
                                  - math.cos(b) * math.acos(max(-1.0, min(1.0, z))))
                f += lens / (4.0 * math.pi)
        return f

    # Haar measure in Rodrigues space: d(mu) = dV / (pi^2 (1 + rho^2)^2); rotation angle <= theta  <=>  rho <= tan(theta/2)
    rho_max = min(math.tan(math.radians(theta) / 2.0) if theta < 180.0 else math.inf, rho_vertex)
    breaks = [0.0] + [x for x in (t8, 1.0 / math.sqrt(3.0), rho_edge) if x < rho_max] + [rho_max]
    nodes, weights = np.polynomial.legendre.leggauss(60)
    s = 0.5 * (nodes + 1.0)
    h = s * s * (3.0 - 2.0 * s)                                 # smoothstep map: removes endpoint singularities
    dh = 6.0 * s * (1.0 - s)
    total = 0.0
    for lo, hi in zip(breaks[:-1], breaks[1:]):
        rho = lo + (hi - lo) * h
        vals = np.array([inside_fraction(x) for x in rho]) * rho ** 2 / (1.0 + rho ** 2) ** 2
        total += 0.5 * (hi - lo) * float(np.sum(weights * vals * dh))
    return min(1.0, 24.0 * 4.0 / math.pi * total)


# Main Problem

## Prompt

Grain-boundary engineering programmes report the fraction of coincidence-site-lattice (CSL) boundaries of each index \\(\\Sigma\\) measured by EBSD, and these fractions are only meaningful in comparison with the fractions expected for a random (untextured) cubic polycrystal. Compute these random-texture reference fractions exactly for Brandon's criterion. All conventions are fixed below.

**Rotations.** Rotations are elements of \\(SO(3)\\), expressed in the cubic crystal axes. The distance \\(d(R_1,R_2)\\in[0^\\circ,180^\\circ]\\) between two rotations is the rotation angle of \\(R_1R_2^{-1}\\), and \\(B(R,r)=\\{Q\\in SO(3):\\ d(Q,R)\\le r\\}\\) is the ball of radius \\(r\\) around \\(R\\). The misorientation \\(M\\) of a random boundary is distributed according to the uniform (Haar) probability measure \\(\\mu\\) on \\(SO(3)\\), normalized such that \\(\\mu(SO(3))=1\\).

**\\(\\Sigma\\) rotations.** A proper rotation matrix \\(R\\) (\\(R^{\\mathsf T}R=I\\), \\(\\det R=+1\\)) is a \\(\\Sigma\\) rotation if \\(\\Sigma\\) is the smallest positive integer for which all nine entries of \\(\\Sigma R\\) are integers. Let \\(S_\\Sigma\\) be the set of all \\(\\Sigma\\) rotations. \\(S_1\\) is the set of the 24 proper rotations of the cube.

**Brandon's criterion.** With the Brandon constant \\(\\theta_0\\), the misorientation \\(M\\) satisfies the criterion for the odd index \\(\\Sigma\\) if \\(M\\in U_\\Sigma=\\bigcup_{R\\in S_\\Sigma}B(R,\\theta_0/\\sqrt{\\Sigma})\\), that is, if \\(M\\) lies within \\(\\theta_0/\\sqrt{\\Sigma}\\) of at least one \\(\\Sigma\\) rotation. For \\(\\Sigma=1\\) this means that the disorientation angle of the boundary is at most \\(\\theta_0\\) (low-angle boundary). A boundary is classified as the smallest odd \\(\\Sigma\\le\\Sigma_{\\max}\\) whose criterion it satisfies, so that the set of boundaries classified as \\(\\Sigma\\) is \\(U_\\Sigma\\setminus\\bigcup_{\\Sigma'<\\Sigma}U_{\\Sigma'}\\) (union over odd \\(\\Sigma'\\)).

Implement the function below. It returns a NumPy array \\(F\\) with one entry for each odd \\(\\Sigma=1,3,5,\\dots\\le\\Sigma_{\\max}\\), in increasing order of \\(\\Sigma\\), where \\(F_k=\\mu\\big(U_\\Sigma\\setminus\\bigcup_{\\Sigma'<\\Sigma}U_{\\Sigma'}\\big)\\) for \\(\\Sigma=2k+1\\) is the probability that a random boundary is classified as \\(\\Sigma\\). The arguments satisfy the following condition: no rotation lies in three or more of the balls \\(B(R,\\theta_0/\\sqrt{\\Sigma})\\) with \\(R\\in S_\\Sigma\\) and odd \\(\\Sigma\\le\\Sigma_{\\max}\\) (each of the 24 balls around the elements of \\(S_1\\) counted separately). Every entry must have an absolute error below \\(10^{-10}\\). Use only NumPy and the Python standard library.

```python
import numpy as np

def random_csl_fractions(sigma_max, theta0=15.0):
    """
    Fractions of random (Haar-distributed) cubic grain boundaries classified as Sigma by Brandon's criterion.

    Inputs:
        sigma_max : int
            Largest CSL index considered (>= 1). Only odd Sigma = 1, 3, 5, ... <= sigma_max are used.
        theta0 : float, default 15.0
            Brandon constant in degrees; the tolerance for index Sigma is theta0 / sqrt(Sigma) degrees.

    Output:
        fractions : numpy.ndarray of float, shape ((sigma_max + 1) // 2,)
            fractions[k] is the probability that a random boundary is classified as Sigma = 2k + 1, i.e. that its
            misorientation lies within theta0 / sqrt(Sigma) of a Sigma rotation but not within theta0 / sqrt(Sigma')
            of any Sigma' rotation with odd Sigma' < Sigma. Absolute error below 1e-10 for every entry.
    """
    ...
```


## Background

**CSL boundaries and Brandon's criterion.** Two cubic lattices related by a rotation \\(R\\) share a coincidence-site lattice only if \\(R\\) has rational entries in the cubic axes, and the index \\(\\Sigma\\) is the smallest positive integer such that \\(\\Sigma R\\) is an integer matrix; only odd indices occur. Brandon proposed that a boundary belongs to \\(\\Sigma\\) if its misorientation deviates from an exact \\(\\Sigma\\) rotation by at most \\(15^\\circ/\\sqrt{\\Sigma}\\). The \\(\\Sigma=1\\) balls are centred on the 24 symmetry rotations of the cube, so \\(U_1\\) is the set of misorientations whose disorientation angle is at most \\(\\theta_0\\). Every set \\(U_\\Sigma\\) is invariant under \\(M\\to O_1MO_2\\) with \\(O_1,O_2\\in S_1\\), so the classification does not depend on the choice among equivalent descriptions of a boundary. Since the tolerance balls of different \\(\\Sigma\\) rotations can overlap, the smallest-index rule assigns every boundary to exactly one class, and the fraction of a class is smaller than the total measure of its balls when they overlap balls of smaller index or each other.

**Measures.** The Haar measure is invariant under multiplication from the left and from the right by fixed rotations. The rotation angle \\(\\omega\\) of a random rotation has the probability density \\((1-\\cos\\omega)/\\pi\\) on \\([0,\\pi]\\), so \\(\\mu(B(R,r))=(r-\\sin r)/\\pi\\) (with \\(r\\) in radians), and the measure of the intersection of two balls depends only on their radii and on the distance between their centres. Rotations correspond two-to-one to unit quaternions \\(\\pm q\\), the Haar measure to the uniform measure on the unit sphere \\(S^3\\), and \\(\\cos\\big(d(R_1,R_2)/2\\big)=|q_1\\cdot q_2|\\).

**\\(\\Sigma\\) rotations.** \\(S_\\Sigma\\) is empty for even \\(\\Sigma\\), is closed under multiplication from either side by the 24 elements of \\(S_1\\), and contains \\(24\\,\\psi(\\Sigma)\\) rotations with \\(\\psi(\\Sigma)=\\Sigma\\prod_{p\\mid\\Sigma}(1+1/p)\\) (product over the distinct prime divisors). For example, \\(S_3\\) contains the \\(60^\\circ\\) rotation about \\([111]\\), and \\(S_5\\) the rotation by \\(2\\arctan(1/3)\\approx36.87^\\circ\\) about \\([100]\\).

**Example.** For \\(\\theta_0=15^\\circ\\) the \\(\\Sigma3\\) balls (radius \\(8.66^\\circ\\)) overlap neither each other nor the low-angle balls, so the random \\(\\Sigma3\\) fraction is \\(96\\,(r-\\sin r)/\\pi\\approx1.76\\,\\%\\) with \\(r=15^\\circ/\\sqrt3\\).

References: D. G. Brandon, Acta Metallurgica 14, 1479-1484 (1966), https://doi.org/10.1016/0001-6160(66)90168-4; J. K. Mackenzie, Biometrika 45, 229-240 (1958), https://doi.org/10.1093/biomet/45.1-2.229; H. Grimmer, W. Bollmann and D. H. Warrington, Acta Cryst. A30, 197-207 (1974), https://doi.org/10.1107/S056773947400043X


## Testing Template

In [ ]:
import math
import numpy as np


def _ball(r_deg):
    r = math.radians(r_deg)
    return (r - math.sin(r)) / math.pi


def _psi(n):
    result, m, p = n, n, 2
    while p * p <= m:
        if m % p == 0:
            result = result // p * (p + 1)
            while m % p == 0:
                m //= p
        p += 1
    if m > 1:
        result = result // m * (m + 1)
    return result


def test_case_1():
    # tolerance balls that do not overlap: every class receives 24 psi(Sigma) full balls
    for sigma_max, theta0 in [(1, 15.0), (3, 15.0), (9, 15.0), (7, 18.0)]:
        result = np.asarray(random_csl_fractions(sigma_max, theta0), dtype=float)
        sigmas = list(range(1, sigma_max + 1, 2))
        expected = np.array([24 * _psi(s) * _ball(theta0 / math.sqrt(s)) for s in sigmas])
        assert result.shape == (len(sigmas),)
        assert np.allclose(result, expected, rtol=0.0, atol=1e-10)
    # an even sigma_max adds no class
    assert np.allclose(random_csl_fractions(10, 15.0), random_csl_fractions(9, 15.0), rtol=0.0, atol=1e-12)


def test_case_2():
    # Sigma 5 balls overlap each other (two Sigma 5 rotations are 16.26 degrees apart)
    cases = [
        (5, 20.0, [0.05382523719728907, 0.04160335142019592, 0.02885140877062986]),
        (5, 22.0, [0.07154976259933674, 0.05535044819155971, 0.03779935765230063]),
    ]
    for sigma_max, theta0, expected in cases:
        result = np.asarray(random_csl_fractions(sigma_max, theta0), dtype=float)
        assert result.shape == (len(expected),)
        assert np.allclose(result, expected, rtol=0.0, atol=1e-10)
    # without overlaps the Sigma 5 class would receive 144 full balls
    assert random_csl_fractions(5, 20.0)[2] < 144 * _ball(20.0 / math.sqrt(5)) - 1e-6


def test_case_3():
    # balls of larger Sigma overlap balls of smaller Sigma (for example Sigma 25 near the low-angle balls)
    cases = [
        (25, 15.0, [0.022768141069261425, 0.0175670158480094, 0.01225221408924306, 0.009863846569198467,
                    0.010150048370006258, 0.00751229987201303, 0.0068220440633065225, 0.009436069499023763,
                    0.005398532980136142, 0.005495128070182801, 0.007454957506814307, 0.004970164661823999,
                    0.005252202188366162]),
        (21, 12.0, [0.011671680628469122, 0.00899801180235907, 0.006274681679761355, 0.005051179616720117,
                    0.005197537194368978, 0.0038467289438533702, 0.003493218054635917, 0.00483166495921801,
                    0.0029755364879893954, 0.002824459952343735, 0.003889212185806869]),
    ]
    for sigma_max, theta0, expected in cases:
        result = np.asarray(random_csl_fractions(sigma_max, theta0), dtype=float)
        assert result.shape == (len(expected),)
        assert np.allclose(result, expected, rtol=0.0, atol=1e-10)
    # the fraction of a class does not depend on classes of larger index
    assert np.allclose(random_csl_fractions(15, 15.0), random_csl_fractions(25, 15.0)[:8], rtol=0.0, atol=1e-12)


## Solution

In [ ]:
import math

import numpy as np


def ball_overlap_measure(q1, q2, r1, r2):
    q1 = np.asarray(q1, dtype=float)
    q2 = np.asarray(q2, dtype=float)
    q1 = q1 / np.linalg.norm(q1)
    q2 = q2 / np.linalg.norm(q2)
    dot = float(q1 @ q2)
    if dot < 0.0:
        q2, dot = -q2, -dot
    # angle between the two quaternions on S^3 (half the rotation angle between the rotations), in [0, pi/2]
    delta = math.atan2(float(np.linalg.norm(q1 - dot * q2)), dot)
    alpha = 0.5 * math.radians(r1)
    beta = 0.5 * math.radians(r2)

    # The uniform measure on S^3 projects onto the uniform measure on the unit disk of any 2-plane. In the plane spanned
    # by q1 and q2, the cap {x.q1 >= cos(alpha)} becomes the disk segment {u >= cos(alpha)} and the cap around +-q2 the
    # segment {u cos(t) + v sin(t) >= cos(beta)} with t = delta (for +q2) or t = pi - delta (for -q2).
    def half_plane_clip(poly, nx, ny, c):  # keep nx*u + ny*v >= c
        out = []
        for k in range(len(poly)):
            p, q = poly[k], poly[(k + 1) % len(poly)]
            fp = nx * p[0] + ny * p[1] - c
            fq = nx * q[0] + ny * q[1] - c
            if fp >= 0.0:
                out.append(p)
            if (fp >= 0.0) != (fq >= 0.0):
                s = fp / (fp - fq)
                out.append((p[0] + s * (q[0] - p[0]), p[1] + s * (q[1] - p[1])))
        return out

    def disk_triangle_area(a, b):  # signed area of (unit disk) ∩ triangle(origin, a, b)
        dx, dy = b[0] - a[0], b[1] - a[1]
        qa = dx * dx + dy * dy
        if qa == 0.0:
            return 0.0
        qb = a[0] * dx + a[1] * dy
        qc = a[0] * a[0] + a[1] * a[1] - 1.0
        disc = qb * qb - qa * qc
        if disc <= 0.0:  # the line does not enter the open disk: the whole piece lies outside
            return 0.5 * math.atan2(a[0] * b[1] - a[1] * b[0], a[0] * b[0] + a[1] * b[1])
        cuts = [0.0]
        if disc > 0.0:
            root = math.sqrt(disc)
            for t in ((-qb - root) / qa, (-qb + root) / qa):
                if 0.0 < t < 1.0:
                    cuts.append(t)
        cuts.append(1.0)
        area = 0.0
        for t0, t1 in zip(cuts[:-1], cuts[1:]):
            p = (a[0] + t0 * dx, a[1] + t0 * dy)
            q = (a[0] + t1 * dx, a[1] + t1 * dy)
            tm = 0.5 * (t0 + t1)
            mx, my = a[0] + tm * dx, a[1] + tm * dy
            cross = p[0] * q[1] - p[1] * q[0]
            if mx * mx + my * my <= 1.0:
                area += 0.5 * cross
            else:
                area += 0.5 * math.atan2(cross, p[0] * q[0] + p[1] * q[1])
        return area

    def region_area(t):
        poly = [(-2.0, -2.0), (2.0, -2.0), (2.0, 2.0), (-2.0, 2.0)]
        poly = half_plane_clip(poly, 1.0, 0.0, math.cos(alpha))
        if poly:
            poly = half_plane_clip(poly, math.cos(t), math.sin(t), math.cos(beta))
        if len(poly) < 3:
            return 0.0
        return abs(sum(disk_triangle_area(poly[k], poly[(k + 1) % len(poly)]) for k in range(len(poly))))

    # both caps around +q1 and -q1 are counted: factor 2; S^3 measure = disk area / pi
    return 2.0 * (region_area(delta) + region_area(math.pi - delta)) / math.pi



def cubic_disorientation_cdf(theta):
    theta = float(theta)
    if theta <= 0.0:
        return 0.0
    if theta >= 180.0:
        return 1.0

    # Fundamental zone of the 24 proper cubic rotations in Rodrigues space r = n tan(omega/2): a truncated cube bounded by
    # |r_i| <= tan(pi/8) (from the 90 degree rotations) and |r_1| + |r_2| + |r_3| <= 1 (from the 120 degree rotations).
    t8 = math.tan(math.pi / 8.0)
    normals, dists = [], []
    for k in range(3):
        for sgn in (1.0, -1.0):
            n = np.zeros(3)
            n[k] = sgn
            normals.append(n)
            dists.append(t8)
    for sx in (1.0, -1.0):
        for sy in (1.0, -1.0):
            for sz in (1.0, -1.0):
                normals.append(np.array([sx, sy, sz]) / math.sqrt(3.0))
                dists.append(1.0 / math.sqrt(3.0))
    normals = np.array(normals)
    dists = np.array(dists)
    cos_pairs = np.clip(normals @ normals.T, -1.0, 1.0)
    rho_edge = math.sqrt(2.0) * t8                              # nearest points of the edges
    rho_vertex = math.sqrt(2.0 * t8 * t8 + (1.0 - 2.0 * t8) ** 2)  # vertices (disorientation 62.80 degrees)

    def inside_fraction(rho):
        # fraction of the sphere |r| = rho inside the zone: caps beyond each face, corrected by the pairwise overlaps of
        # caps (edges); for rho below the vertex distance no point of the sphere lies beyond three faces
        if rho >= rho_vertex:
            return 0.0
        cap = np.arccos(np.clip(dists / rho, -1.0, 1.0))        # angular radii of the caps (0 if the face is not reached)
        f = 1.0 - np.sum(0.5 * (1.0 - np.cos(cap)))
        m = len(dists)
        for i in range(m):
            for j in range(i + 1, m):
                a, b = cap[i], cap[j]
                if a <= 0.0 or b <= 0.0:
                    continue
                c = math.acos(cos_pairs[i, j])
                if c >= a + b:
                    continue
                if c <= abs(a - b):
                    lens = 2.0 * math.pi * (1.0 - math.cos(min(a, b)))
                else:
                    x = (math.cos(c) - math.cos(a) * math.cos(b)) / (math.sin(a) * math.sin(b))
                    y = (math.cos(b) - math.cos(c) * math.cos(a)) / (math.sin(c) * math.sin(a))
                    z = (math.cos(a) - math.cos(c) * math.cos(b)) / (math.sin(c) * math.sin(b))
                    lens = 2.0 * (math.pi - math.acos(max(-1.0, min(1.0, x)))
                                  - math.cos(a) * math.acos(max(-1.0, min(1.0, y)))
                                  - math.cos(b) * math.acos(max(-1.0, min(1.0, z))))
                f += lens / (4.0 * math.pi)
        return f

    # Haar measure in Rodrigues space: d(mu) = dV / (pi^2 (1 + rho^2)^2); rotation angle <= theta  <=>  rho <= tan(theta/2)
    rho_max = min(math.tan(math.radians(theta) / 2.0) if theta < 180.0 else math.inf, rho_vertex)
    breaks = [0.0] + [x for x in (t8, 1.0 / math.sqrt(3.0), rho_edge) if x < rho_max] + [rho_max]
    nodes, weights = np.polynomial.legendre.leggauss(60)
    s = 0.5 * (nodes + 1.0)
    h = s * s * (3.0 - 2.0 * s)                                 # smoothstep map: removes endpoint singularities
    dh = 6.0 * s * (1.0 - s)
    total = 0.0
    for lo, hi in zip(breaks[:-1], breaks[1:]):
        rho = lo + (hi - lo) * h
        vals = np.array([inside_fraction(x) for x in rho]) * rho ** 2 / (1.0 + rho ** 2) ** 2
        total += 0.5 * (hi - lo) * float(np.sum(weights * vals * dh))
    return min(1.0, 24.0 * 4.0 / math.pi * total)


def random_csl_fractions(sigma_max, theta0=15.0):
    sigma_max = int(sigma_max)

    def sigma_rotations(sigma):
        # all proper rotations R with sigma the smallest positive integer making sigma * R integral (odd sigma only)
        if sigma == 1:
            r = np.arange(-1, 2)
        else:
            r = np.arange(-sigma, sigma + 1)
        x, y, z = np.meshgrid(r, r, r, indexing="ij")
        mask = x * x + y * y + z * z == sigma * sigma
        rows = np.stack([x[mask], y[mask], z[mask]], axis=1).astype(np.int64)
        i_idx, j_idx = np.nonzero(rows @ rows.T == 0)
        r1, r2 = rows[i_idx], rows[j_idx]
        cross = np.cross(r1, r2)
        ok = np.all(cross % sigma == 0, axis=1)
        mats = np.stack([r1[ok], r2[ok], cross[ok] // sigma], axis=1)
        common = np.gcd.reduce(mats.reshape(len(mats), 9), axis=1)
        mats = mats[np.gcd(common, sigma) == 1].astype(float) / sigma
        # rotation matrices -> unit quaternions (w, x, y, z), Shepperd's method (stable for all angles)
        quats = []
        for m in mats:
            tr = m[0, 0] + m[1, 1] + m[2, 2]
            k = int(np.argmax([tr, m[0, 0], m[1, 1], m[2, 2]]))
            if k == 0:
                s = 2.0 * math.sqrt(1.0 + tr)
                q = [0.25 * s, (m[2, 1] - m[1, 2]) / s, (m[0, 2] - m[2, 0]) / s, (m[1, 0] - m[0, 1]) / s]
            elif k == 1:
                s = 2.0 * math.sqrt(1.0 + m[0, 0] - m[1, 1] - m[2, 2])
                q = [(m[2, 1] - m[1, 2]) / s, 0.25 * s, (m[0, 1] + m[1, 0]) / s, (m[0, 2] + m[2, 0]) / s]
            elif k == 2:
                s = 2.0 * math.sqrt(1.0 - m[0, 0] + m[1, 1] - m[2, 2])
                q = [(m[0, 2] - m[2, 0]) / s, (m[0, 1] + m[1, 0]) / s, 0.25 * s, (m[1, 2] + m[2, 1]) / s]
            else:
                s = 2.0 * math.sqrt(1.0 - m[0, 0] - m[1, 1] + m[2, 2])
                q = [(m[1, 0] - m[0, 1]) / s, (m[0, 2] + m[2, 0]) / s, (m[1, 2] + m[2, 1]) / s, 0.25 * s]
            q = np.array(q)
            quats.append(q / np.linalg.norm(q))
        return np.array(quats)

    sigmas = list(range(1, sigma_max + 1, 2))
    centres, radii, labels = [], [], []
    for sigma in sigmas:
        q = sigma_rotations(sigma)
        if len(q) == 0:
            continue
        centres.append(q)
        radii.append(np.full(len(q), theta0 / math.sqrt(sigma)))
        labels.append(np.full(len(q), sigma))
    centres = np.concatenate(centres)
    radii = np.concatenate(radii)
    labels = np.concatenate(labels)

    fractions = np.zeros(len(sigmas))
    for k, sigma in enumerate(sigmas):
        own = labels == sigma
        if not np.any(own):
            continue
        r = math.radians(theta0 / math.sqrt(sigma))
        if sigma == 1:
            fractions[k] = cubic_disorientation_cdf(theta0)
            continue
        value = own.sum() * (r - math.sin(r)) / math.pi
        # overlaps with balls of the same index (counted once per pair) and with balls of any smaller index
        own_idx = np.nonzero(own)[0]
        other_idx = np.nonzero(labels <= sigma)[0]
        dots = np.abs(centres[own_idx] @ centres[other_idx].T)
        limit = np.cos(np.radians(radii[own_idx][:, None] + radii[other_idx][None, :]) / 2.0)
        ii, jj = np.nonzero(dots > limit)
        for a, b in zip(own_idx[ii], other_idx[jj]):
            if a == b or (labels[b] == sigma and b < a):
                continue
            value -= ball_overlap_measure(centres[a], centres[b], radii[a], radii[b])
        fractions[k] = value
    return fractions
